###1) Konfiguracja

In [0]:
from datetime import (
    datetime,
    date,
    time,
    timezone,
    timedelta,
)
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)

dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

dbutils.widgets.text(
    "fmm_history_buffer_days",
    "1"
)

EVALUATION_START_DATE_TEXT = dbutils.widgets.get(
    "evaluation_start_date_local"
)

EVALUATION_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

FMM_HISTORY_BUFFER_DAYS = int(
    dbutils.widgets.get(
        "fmm_history_buffer_days"
    )
)

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

evaluation_start_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_START_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

evaluation_end_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_END_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

if evaluation_end_local <= evaluation_start_local:
    raise ValueError(
        "Data końcowa musi być późniejsza "
        "od daty początkowej."
    )

EVALUATION_START_UTC = (
    evaluation_start_local
    .astimezone(timezone.utc)
)

EVALUATION_END_UTC = (
    evaluation_end_local
    .astimezone(timezone.utc)
)

FMM_START_UTC = (
    EVALUATION_START_UTC
    - timedelta(
        days=FMM_HISTORY_BUFFER_DAYS
    )
)

SOURCE_TABLE = (
    "workspace.energy_bronze."
    "caiso_lmp_raw"
)

DAM_TARGET_TABLE = (
    "workspace.energy_silver."
    "caiso_dam_lmp"
)

FMM_TARGET_TABLE = (
    "workspace.energy_silver."
    "caiso_fmm_lmp"
)

NODE_ID = "TH_NP15_GEN-APND"

EXPECTED_DAM_ROWS = int(
    (
        EVALUATION_END_UTC
        - EVALUATION_START_UTC
    ).total_seconds()
    / 3600
)

EXPECTED_FMM_ROWS = int(
    (
        EVALUATION_END_UTC
        - FMM_START_UTC
    ).total_seconds()
    / (15 * 60)
)

COMPONENT_BALANCE_TOLERANCE = 0.02
PRICE_SPIKE_THRESHOLD = 250.0

if not spark.catalog.tableExists(
    SOURCE_TABLE
):
    raise RuntimeError(
        f"Brak tabeli źródłowej: "
        f"{SOURCE_TABLE}"
    )

source_row_count = (
    spark.table(SOURCE_TABLE).count()
)

print(f"Źródło: {SOURCE_TABLE}")

print(
    "Liczba rekordów Bronze:",
    source_row_count
)

print(
    "Oczekiwane rekordy Silver DAM:",
    EXPECTED_DAM_ROWS
)

print(
    "Oczekiwane rekordy Silver FMM:",
    EXPECTED_FMM_ROWS
)

print(f"Tabela DAM: {DAM_TARGET_TABLE}")
print(f"Tabela FMM: {FMM_TARGET_TABLE}")

2# Parsowanie i pivot składników ceny 

In [0]:
component_names = [
    "LMP_PRC",
    "LMP_ENE_PRC",
    "LMP_CONG_PRC",
    "LMP_LOSS_PRC",
    "LMP_GHG_PRC",
]


parsed_prices_df = (
    spark.table(SOURCE_TABLE)
    .withColumn(
        "interval_start_utc",
        F.to_timestamp(
            "interval_starttime_gmt_raw",
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    )
    .withColumn(
        "interval_end_utc",
        F.to_timestamp(
            "interval_endtime_gmt_raw",
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    )
    .withColumn(
        "price_value",
        F.expr(
            "TRY_CAST("
            "price_value_raw AS DOUBLE)"
        )
    )
)


group_columns = [
    "market_type",
    "market_run_id",
    "interval_start_utc",
    "interval_end_utc",
    "opr_dt_raw",
    "opr_hr_raw",
    "opr_interval_raw",
    "node_id_xml",
    "node_id",
    "node",
    "pnode_resmrid",
    "grp_type",
    "raw_value_column",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_csv_name",
    "_dataset_name",
    "_source_system",
    "_requested_node",
    "_ingestion_run_id",
]


wide_prices_df = (
    parsed_prices_df
    .groupBy(
        *group_columns
    )
    .pivot(
        "xml_data_item",
        component_names
    )
    .agg(
        F.first(
            "price_value",
            ignorenulls=True
        )
    )
    .withColumnRenamed(
        "LMP_PRC",
        "lmp_price_usd_mwh"
    )
    .withColumnRenamed(
        "LMP_ENE_PRC",
        "energy_component_usd_mwh"
    )
    .withColumnRenamed(
        "LMP_CONG_PRC",
        "congestion_component_usd_mwh"
    )
    .withColumnRenamed(
        "LMP_LOSS_PRC",
        "loss_component_usd_mwh"
    )
    .withColumnRenamed(
        "LMP_GHG_PRC",
        "ghg_component_usd_mwh"
    )
)


price_component_columns = [
    "lmp_price_usd_mwh",
    "energy_component_usd_mwh",
    "congestion_component_usd_mwh",
    "loss_component_usd_mwh",
    "ghg_component_usd_mwh",
]


component_count_expression = F.lit(0)

for column_name in price_component_columns:
    component_count_expression = (
        component_count_expression
        + F.when(
            F.col(column_name).isNotNull(),
            F.lit(1)
        ).otherwise(
            F.lit(0)
        )
    )


wide_prices_df = (
    wide_prices_df
    .withColumn(
        "source_component_count",
        component_count_expression
    )
    .withColumn(
        "component_sum_usd_mwh",
        F.col(
            "energy_component_usd_mwh"
        )
        + F.col(
            "congestion_component_usd_mwh"
        )
        + F.col(
            "loss_component_usd_mwh"
        )
        + F.col(
            "ghg_component_usd_mwh"
        )
    )
    .withColumn(
        "component_balance_error_usd_mwh",
        F.col("lmp_price_usd_mwh")
        - F.col("component_sum_usd_mwh")
    )
    .withColumn(
        "is_negative_lmp",
        F.col("lmp_price_usd_mwh") < 0
    )
    .withColumn(
        "is_price_spike",
        F.col("lmp_price_usd_mwh")
        > F.lit(PRICE_SPIKE_THRESHOLD)
    )
    .withColumn(
        "quality_status",
        F.when(
            F.col(
                "lmp_price_usd_mwh"
            ).isNull(),
            F.lit("MISSING_LMP")
        )
        .when(
            F.col(
                "source_component_count"
            ) != 5,
            F.lit("MISSING_COMPONENT")
        )
        .when(
            F.abs(
                F.col(
                    "component_balance_error_usd_mwh"
                )
            ) > F.lit(
                COMPONENT_BALANCE_TOLERANCE
            ),
            F.lit("COMPONENT_MISMATCH")
        )
        .otherwise(
            F.lit("VALID")
        )
    )
)


print(
    "Liczba interwałów po pivot: "
    f"{wide_prices_df.count()}"
)

display(
    wide_prices_df
    .select(
        "market_type",
        "interval_start_utc",
        "lmp_price_usd_mwh",
        "energy_component_usd_mwh",
        "congestion_component_usd_mwh",
        "loss_component_usd_mwh",
        "ghg_component_usd_mwh",
        "component_balance_error_usd_mwh",
        "quality_status"
    )
    .orderBy(
        "interval_start_utc",
        "market_type"
    )
    .limit(20)
)

3# Tabela Silver DAM

In [0]:
dam_base_df = (
    wide_prices_df
    .filter(
        F.col("market_type") == "DAM"
    )
    .withColumn(
        "interval_start_local",
        F.from_utc_timestamp(
            "interval_start_utc",
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "interval_end_local",
        F.from_utc_timestamp(
            "interval_end_utc",
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "delivery_date_local",
        F.to_date(
            "interval_start_local"
        )
    )
    .withColumn(
        "delivery_hour_local",
        F.hour(
            "interval_start_local"
        )
    )
    .withColumn(
        "_previous_local_date",
        F.date_sub(
            "delivery_date_local",
            1
        )
    )
    .withColumn(
        "_dam_bid_gate_local",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.col(
                    "_previous_local_date"
                ).cast("string"),
                F.lit("10:00:00")
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "_dam_publication_local",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.col(
                    "_previous_local_date"
                ).cast("string"),
                F.lit("13:00:00")
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "dam_bid_gate_close_at_utc",
        F.to_utc_timestamp(
            "_dam_bid_gate_local",
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "price_available_at_utc",
        F.to_utc_timestamp(
            "_dam_publication_local",
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "interval_duration_minutes",
        (
            F.col("interval_end_utc")
            .cast("long")
            - F.col("interval_start_utc")
            .cast("long")
        ) / F.lit(60.0)
    )
    .withColumn(
        "price_unit",
        F.lit("USD/MWh")
    )
    .withColumn(
        "availability_rule",
        F.lit(
            "DAM result assumed available "
            "at 13:00 America/Los_Angeles "
            "on the previous day"
        )
    )
    .drop(
        "_previous_local_date",
        "_dam_bid_gate_local",
        "_dam_publication_local"
    )
)


dam_hash_expression = F.concat_ws(
    "||",
    F.lit("DAM"),
    F.col("interval_start_utc").cast(
        "string"
    ),
    F.col("interval_end_utc").cast(
        "string"
    ),
    F.coalesce(
        F.col("node_id"),
        F.lit("<NULL>")
    ),
    F.coalesce(
        F.col(
            "lmp_price_usd_mwh"
        ).cast("string"),
        F.lit("<NULL>")
    )
)


caiso_dam_silver_df = (
    dam_base_df
    .withColumn(
        "_transformed_at_utc",
        F.current_timestamp()
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            dam_hash_expression,
            256
        )
    )
    .select(
        "market_type",
        "market_run_id",
        "interval_start_utc",
        "interval_end_utc",
        "interval_start_local",
        "interval_end_local",
        "delivery_date_local",
        "delivery_hour_local",
        "dam_bid_gate_close_at_utc",
        "price_available_at_utc",
        "interval_duration_minutes",
        "node_id",
        "pnode_resmrid",
        "lmp_price_usd_mwh",
        "energy_component_usd_mwh",
        "congestion_component_usd_mwh",
        "loss_component_usd_mwh",
        "ghg_component_usd_mwh",
        "component_sum_usd_mwh",
        "component_balance_error_usd_mwh",
        "source_component_count",
        "price_unit",
        "is_negative_lmp",
        "is_price_spike",
        "quality_status",
        "availability_rule",
        "_source_file_path",
        "_source_file_name",
        "_source_file_size",
        "_source_csv_name",
        "_source_system",
        "_requested_node",
        "_ingestion_run_id",
        "_transformed_at_utc",
        "_record_hash"
    )
)


print(
    "Liczba rekordów Silver DAM: "
    f"{caiso_dam_silver_df.count()}"
)

display(
    caiso_dam_silver_df
    .orderBy("interval_start_utc")
    .limit(10)
)

4# Tabela Silver FMM

In [0]:
fmm_base_df = (
    wide_prices_df
    .filter(
        F.col("market_type") == "FMM"
    )
    .withColumn(
        "interval_start_local",
        F.from_utc_timestamp(
            "interval_start_utc",
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "interval_end_local",
        F.from_utc_timestamp(
            "interval_end_utc",
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "delivery_date_local",
        F.to_date(
            "interval_start_local"
        )
    )
    .withColumn(
        "delivery_hour_local",
        F.hour(
            "interval_start_local"
        )
    )
    .withColumn(
        "delivery_quarter",
        (
            F.minute(
                "interval_start_local"
            ) / F.lit(15)
        ).cast("integer") + F.lit(1)
    )
    .withColumn(
        "rt_bid_gate_close_at_utc",
        (
            F.date_trunc(
                "hour",
                F.col("interval_start_utc")
            ).cast("long")
            - F.lit(75 * 60)
        ).cast("timestamp")
    )
    .withColumn(
        "price_available_at_utc",
        (
            F.col("interval_start_utc")
            .cast("long")
            - F.lit(22 * 60 + 30)
        ).cast("timestamp")
    )
    .withColumn(
        "interval_duration_minutes",
        (
            F.col("interval_end_utc")
            .cast("long")
            - F.col("interval_start_utc")
            .cast("long")
        ) / F.lit(60.0)
    )
    .withColumn(
        "price_unit",
        F.lit("USD/MWh")
    )
    .withColumn(
        "availability_rule",
        F.lit(
            "FMM result assumed available "
            "22.5 minutes before delivery"
        )
    )
)


fmm_hash_expression = F.concat_ws(
    "||",
    F.lit("FMM"),
    F.col("interval_start_utc").cast(
        "string"
    ),
    F.col("interval_end_utc").cast(
        "string"
    ),
    F.coalesce(
        F.col("node_id"),
        F.lit("<NULL>")
    ),
    F.coalesce(
        F.col(
            "lmp_price_usd_mwh"
        ).cast("string"),
        F.lit("<NULL>")
    )
)


caiso_fmm_silver_df = (
    fmm_base_df
    .withColumn(
        "_transformed_at_utc",
        F.current_timestamp()
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            fmm_hash_expression,
            256
        )
    )
    .select(
        "market_type",
        "market_run_id",
        "interval_start_utc",
        "interval_end_utc",
        "interval_start_local",
        "interval_end_local",
        "delivery_date_local",
        "delivery_hour_local",
        "delivery_quarter",
        "rt_bid_gate_close_at_utc",
        "price_available_at_utc",
        "interval_duration_minutes",
        "node_id",
        "pnode_resmrid",
        "lmp_price_usd_mwh",
        "energy_component_usd_mwh",
        "congestion_component_usd_mwh",
        "loss_component_usd_mwh",
        "ghg_component_usd_mwh",
        "component_sum_usd_mwh",
        "component_balance_error_usd_mwh",
        "source_component_count",
        "price_unit",
        "is_negative_lmp",
        "is_price_spike",
        "quality_status",
        "availability_rule",
        "_source_file_path",
        "_source_file_name",
        "_source_file_size",
        "_source_csv_name",
        "_source_system",
        "_requested_node",
        "_ingestion_run_id",
        "_transformed_at_utc",
        "_record_hash"
    )
)


print(
    "Liczba rekordów Silver FMM: "
    f"{caiso_fmm_silver_df.count()}"
)

display(
    caiso_fmm_silver_df
    .orderBy("interval_start_utc")
    .limit(10)
)

5# Zapis tabel Delta Silver

In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "workspace.energy_silver"
)


(
    caiso_dam_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        DAM_TARGET_TABLE
    )
)


(
    caiso_fmm_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        FMM_TARGET_TABLE
    )
)


print(
    f"Zapisano: {DAM_TARGET_TABLE}"
)

print(
    "Liczba DAM: "
    f"{spark.table(DAM_TARGET_TABLE).count()}"
)

print(
    f"Zapisano: {FMM_TARGET_TABLE}"
)

print(
    "Liczba FMM: "
    f"{spark.table(FMM_TARGET_TABLE).count()}"
)

6# Profil cen Silver

In [0]:
silver_price_profile_df = spark.sql(
    f"""
    SELECT
        'DAM' AS market,
        COUNT(*) AS row_count,
        MIN(interval_start_utc)
            AS min_interval_start_utc,
        MAX(interval_start_utc)
            AS max_interval_start_utc,
        MIN(lmp_price_usd_mwh)
            AS minimum_lmp,
        AVG(lmp_price_usd_mwh)
            AS average_lmp,
        MAX(lmp_price_usd_mwh)
            AS maximum_lmp,
        SUM(
            CASE
                WHEN is_negative_lmp
                THEN 1 ELSE 0
            END
        ) AS negative_price_count,
        SUM(
            CASE
                WHEN is_price_spike
                THEN 1 ELSE 0
            END
        ) AS price_spike_count,
        SUM(
            CASE
                WHEN quality_status != 'VALID'
                THEN 1 ELSE 0
            END
        ) AS invalid_quality_count
    FROM {DAM_TARGET_TABLE}

    UNION ALL

    SELECT
        'FMM' AS market,
        COUNT(*) AS row_count,
        MIN(interval_start_utc)
            AS min_interval_start_utc,
        MAX(interval_start_utc)
            AS max_interval_start_utc,
        MIN(lmp_price_usd_mwh)
            AS minimum_lmp,
        AVG(lmp_price_usd_mwh)
            AS average_lmp,
        MAX(lmp_price_usd_mwh)
            AS maximum_lmp,
        SUM(
            CASE
                WHEN is_negative_lmp
                THEN 1 ELSE 0
            END
        ) AS negative_price_count,
        SUM(
            CASE
                WHEN is_price_spike
                THEN 1 ELSE 0
            END
        ) AS price_spike_count,
        SUM(
            CASE
                WHEN quality_status != 'VALID'
                THEN 1 ELSE 0
            END
        ) AS invalid_quality_count
    FROM {FMM_TARGET_TABLE}

    ORDER BY market
    """
)


display(silver_price_profile_df)

7# Kontrola dostępności cen

In [0]:
dam_availability_preview = (
    spark.table(DAM_TARGET_TABLE)
    .select(
        "market_type",
        "interval_start_utc",
        "dam_bid_gate_close_at_utc",
        "price_available_at_utc",
        "lmp_price_usd_mwh"
    )
    .orderBy("interval_start_utc")
    .limit(8)
)


fmm_availability_preview = (
    spark.table(FMM_TARGET_TABLE)
    .select(
        "market_type",
        "interval_start_utc",
        "rt_bid_gate_close_at_utc",
        "price_available_at_utc",
        "lmp_price_usd_mwh"
    )
    .orderBy("interval_start_utc")
    .limit(8)
)


print("DAM:")
display(dam_availability_preview)

print("FMM:")
display(fmm_availability_preview)

8# Testy Silver CAISO

In [0]:
dam_test_df = spark.table(
    DAM_TARGET_TABLE
)

fmm_test_df = spark.table(
    FMM_TARGET_TABLE
)


dam_row_count = dam_test_df.count()
fmm_row_count = fmm_test_df.count()


dam_duplicate_count = (
    dam_test_df
    .groupBy(
        "interval_start_utc",
        "node_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


fmm_duplicate_count = (
    fmm_test_df
    .groupBy(
        "interval_start_utc",
        "node_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


dam_invalid_quality_count = (
    dam_test_df
    .filter(
        F.col("quality_status") != "VALID"
    )
    .count()
)


fmm_invalid_quality_count = (
    fmm_test_df
    .filter(
        F.col("quality_status") != "VALID"
    )
    .count()
)


dam_component_mismatch_count = (
    dam_test_df
    .filter(
        F.abs(
            F.col(
                "component_balance_error_usd_mwh"
            )
        ) > F.lit(
            COMPONENT_BALANCE_TOLERANCE
        )
    )
    .count()
)


fmm_component_mismatch_count = (
    fmm_test_df
    .filter(
        F.abs(
            F.col(
                "component_balance_error_usd_mwh"
            )
        ) > F.lit(
            COMPONENT_BALANCE_TOLERANCE
        )
    )
    .count()
)


dam_availability_violation_count = (
    dam_test_df
    .filter(
        F.col("price_available_at_utc")
        >= F.col("interval_start_utc")
    )
    .count()
)


fmm_availability_violation_count = (
    fmm_test_df
    .filter(
        F.col("price_available_at_utc")
        >= F.col("interval_start_utc")
    )
    .count()
)


dam_price_known_at_bid_gate_count = (
    dam_test_df
    .filter(
        F.col("price_available_at_utc")
        <= F.col(
            "dam_bid_gate_close_at_utc"
        )
    )
    .count()
)


fmm_price_known_at_bid_gate_count = (
    fmm_test_df
    .filter(
        F.col("price_available_at_utc")
        <= F.col(
            "rt_bid_gate_close_at_utc"
        )
    )
    .count()
)


dam_wrong_duration_count = (
    dam_test_df
    .filter(
        F.col(
            "interval_duration_minutes"
        ) != 60
    )
    .count()
)


fmm_wrong_duration_count = (
    fmm_test_df
    .filter(
        F.col(
            "interval_duration_minutes"
        ) != 15
    )
    .count()
)


dam_wrong_node_count = (
    dam_test_df
    .filter(
        F.col("node_id") != NODE_ID
    )
    .count()
)


fmm_wrong_node_count = (
    fmm_test_df
    .filter(
        F.col("node_id") != NODE_ID
    )
    .count()
)


test_results = {
    "dam_row_count":
        dam_row_count,
    "fmm_row_count":
        fmm_row_count,
    "dam_duplicate_count":
        dam_duplicate_count,
    "fmm_duplicate_count":
        fmm_duplicate_count,
    "dam_invalid_quality_count":
        dam_invalid_quality_count,
    "fmm_invalid_quality_count":
        fmm_invalid_quality_count,
    "dam_component_mismatch_count":
        dam_component_mismatch_count,
    "fmm_component_mismatch_count":
        fmm_component_mismatch_count,
    "dam_availability_violation_count":
        dam_availability_violation_count,
    "fmm_availability_violation_count":
        fmm_availability_violation_count,
    "dam_price_known_at_bid_gate_count":
        dam_price_known_at_bid_gate_count,
    "fmm_price_known_at_bid_gate_count":
        fmm_price_known_at_bid_gate_count,
    "dam_wrong_duration_count":
        dam_wrong_duration_count,
    "fmm_wrong_duration_count":
        fmm_wrong_duration_count,
    "dam_wrong_node_count":
        dam_wrong_node_count,
    "fmm_wrong_node_count":
        fmm_wrong_node_count,
}


for name, value in test_results.items():
    print(f"{name}: {value}")


errors = []


if dam_row_count != EXPECTED_DAM_ROWS:
    errors.append(
        "Niepoprawna liczba DAM."
    )

if fmm_row_count != EXPECTED_FMM_ROWS:
    errors.append(
        "Niepoprawna liczba FMM."
    )

zero_expected_tests = {
    "dam_duplicate_count":
        dam_duplicate_count,
    "fmm_duplicate_count":
        fmm_duplicate_count,
    "dam_invalid_quality_count":
        dam_invalid_quality_count,
    "fmm_invalid_quality_count":
        fmm_invalid_quality_count,
    "dam_component_mismatch_count":
        dam_component_mismatch_count,
    "fmm_component_mismatch_count":
        fmm_component_mismatch_count,
    "dam_availability_violation_count":
        dam_availability_violation_count,
    "fmm_availability_violation_count":
        fmm_availability_violation_count,
    "dam_price_known_at_bid_gate_count":
        dam_price_known_at_bid_gate_count,
    "fmm_price_known_at_bid_gate_count":
        fmm_price_known_at_bid_gate_count,
    "dam_wrong_duration_count":
        dam_wrong_duration_count,
    "fmm_wrong_duration_count":
        fmm_wrong_duration_count,
    "dam_wrong_node_count":
        dam_wrong_node_count,
    "fmm_wrong_node_count":
        fmm_wrong_node_count,
}


for name, value in (
    zero_expected_tests.items()
):
    if value != 0:
        errors.append(
            f"{name} powinno wynosić 0, "
            f"a wynosi {value}."
        )


if errors:
    print()
    print("TESTY NIEZALICZONE:")

    for error in errors:
        print(f"- {error}")

    raise RuntimeError(
        "Testy Silver CAISO "
        "nie zostały zaliczone."
    )


print()
print(
    "Wszystkie testy Silver CAISO "
    "zostały zaliczone."
)